# Notebook 3 — SEIRD Simulation
Runs 5 simulations per graph (10 graphs × 5 runs = 50 training pairs).
Run on **Colab GPU** (T4). Runtime ~60-90 minutes total.
Output: `training_pairs.pt` — list of (x0, y, graph_seed, pop_seed) tuples.

In [ ]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/epidemic_project'
print('Graphs available:')
for f in sorted(os.listdir(f'{BASE}/graphs')):
    print(' ', f)

In [ ]:
# Cell 2 — Install
!pip install -q torch-geometric torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-2.2.0+cu118.html

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 3 — Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print('Imports OK')

In [ ]:
# Cell 4 — SEIRD simulator
def compute_node_params(df, beta_base, device):
    """Compute per-node SEIRD parameters from population attributes."""
    age_norm    = torch.tensor(df['age'].values / 95.0,      dtype=torch.float, device=device)
    comorbidity = torch.tensor(df['comorbidity'].values,      dtype=torch.float, device=device)
    vaccinated  = torch.tensor(df['vaccinated'].values,       dtype=torch.float, device=device)

    GAMMA_BASE = 0.08
    DELTA_BASE = 0.003

    beta_i  = beta_base  * (1 - 0.4 * vaccinated) * (1 + 0.3 * comorbidity)
    gamma_i = GAMMA_BASE * (1 - 0.4 * comorbidity) * (1 + 0.1 * (age_norm < 0.3).float())
    delta_i = DELTA_BASE * (1 + 3.0 * comorbidity) * (1 + 2.0 * (age_norm > 0.7).float())

    return beta_i, gamma_i, delta_i


def simulate_seird(graph, df, seed_nodes, beta_base=0.25,
                   sigma=0.20, imm_wane=0.002, T=60, device='cpu'):
    """
    Run one stochastic SEIRD simulation.
    Returns tensor of shape (T, N, 5).
    """
    N          = graph.num_nodes
    edge_index = graph.edge_index.to(device)
    edge_attr  = graph.edge_attr.squeeze(1).to(device)
    src        = edge_index[0]
    dst        = edge_index[1]

    beta_i, gamma_i, delta_i = compute_node_params(df, beta_base, device)

    # Initial state
    state = torch.zeros(N, 5, device=device)
    state[:, 0] = 1.0
    for s in seed_nodes:
        if s < N:
            state[s, 0] = 0.0
            state[s, 2] = 1.0

    snapshots = []

    for t in range(T):
        snapshots.append(state.clone().cpu())

        p_I     = state[:, 2]
        contrib = torch.log(1.0 - beta_i[dst] * edge_attr * p_I[src] + 1e-8)

        log_escape = torch.zeros(N, device=device)
        log_escape.scatter_add_(0, dst, contrib)
        p_escape = torch.exp(log_escape)

        delta_S      = state[:, 0] * (1 - p_escape)
        delta_E_to_I = state[:, 1] * sigma
        delta_I_to_R = state[:, 2] * gamma_i
        delta_I_to_D = state[:, 2] * delta_i
        delta_R_to_S = state[:, 3] * imm_wane

        new_state = torch.zeros_like(state)
        new_state[:, 0] = (state[:, 0] - delta_S      + delta_R_to_S).clamp(0)
        new_state[:, 1] = (state[:, 1] + delta_S      - delta_E_to_I).clamp(0)
        new_state[:, 2] = (state[:, 2] + delta_E_to_I - delta_I_to_R - delta_I_to_D).clamp(0)
        new_state[:, 3] = (state[:, 3] + delta_I_to_R - delta_R_to_S).clamp(0)
        new_state[:, 4] = (state[:, 4] + delta_I_to_D).clamp(0)

        row_sums  = new_state.sum(dim=1, keepdim=True).clamp(min=1e-8)
        new_state = new_state / row_sums

        dead_mask = new_state[:, 4] > 0.95
        new_state[dead_mask]     = 0.0
        new_state[dead_mask, 4] = 1.0

        state = new_state

    return torch.stack(snapshots, dim=0)  # (T, N, 5)

In [ ]:
# Cell 5 — Choose diverse seed sets
# For each graph, we use 3 DIFFERENT seeding strategies:
# 1. Top-degree nodes (super-spreaders)
# 2. Random high-contact nodes
# 3. A specific zone cluster
# 4. Mixed zones
# 5. Low-degree nodes (slow spread scenario)
# This forces the model to learn spread dynamics, not just 'what happens in Pune'

def get_seed_sets(graph, df, n_sets=5):
    """
    Return n_sets different lists of seed node IDs for varied simulations.
    """
    N   = graph.num_nodes
    src = graph.edge_index[0]
    degree = torch.zeros(N, dtype=torch.long)
    degree.scatter_add_(0, src, torch.ones(src.shape[0], dtype=torch.long))

    social_freq = torch.tensor(df['social_freq'].values, dtype=torch.float)

    # Strategy 1: top-degree seeds (super-spreaders)
    top_deg  = torch.topk(degree.float(), k=300).indices.numpy()
    seeds1   = np.random.choice(top_deg, size=10, replace=False).tolist()

    # Strategy 2: high social_freq seeds (random sample from top 20%)
    top_soc  = torch.topk(social_freq, k=int(0.2*N)).indices.numpy()
    seeds2   = np.random.choice(top_soc, size=10, replace=False).tolist()

    # Strategy 3: single zone cluster (seeds all in one zone)
    zone_id  = np.random.choice(range(1, 21))
    zone_idx = df[df['zone'] == zone_id].index.values
    seeds3   = np.random.choice(zone_idx, size=min(8, len(zone_idx)), replace=False).tolist()

    # Strategy 4: spread across multiple zones
    zones    = np.random.choice(range(1, 21), size=5, replace=False)
    seeds4   = []
    for z in zones:
        zi = df[df['zone'] == z].index.values
        if len(zi) > 0:
            seeds4.append(int(np.random.choice(zi)))

    # Strategy 5: low-degree nodes (slow-burning spread)
    bot_deg  = torch.topk(-degree.float(), k=300).indices.numpy()
    seeds5   = np.random.choice(bot_deg, size=8, replace=False).tolist()

    return [seeds1, seeds2, seeds3, seeds4, seeds5]

In [ ]:
# Cell 6 — Run simulations for all 10 graphs
SEEDS       = [0, 7, 13, 21, 42, 55, 77, 88, 99, 111]
# Vary beta slightly across graphs for more dynamics diversity
BETA_VALUES = [0.40, 0.50, 0.45, 0.55, 0.45, 0.38, 0.58, 0.48, 0.42, 0.50]

all_pairs = []   # list of dicts: {x0, y, graph_seed, sim_idx, beta}

for seed, beta in zip(SEEDS, BETA_VALUES):
    print(f'\n=== Graph seed={seed}, beta={beta} ===')

    graph = torch.load(f'{BASE}/graphs/graph_seed{seed}.pt')
    df    = pd.read_parquet(f'{BASE}/data/population_seed{seed}.parquet')

    seed_sets = get_seed_sets(graph, df, n_sets=5)

    for sim_idx, seeds in enumerate(seed_sets):
        print(f'  Sim {sim_idx+1}/5 ({len(seeds)} seed nodes)...', end=' ')

        np.random.seed(seed * 100 + sim_idx)   # reproducible

        snaps = simulate_seird(
            graph, df, seeds,
            beta_base=beta,
            device=str(device), T=60
        )  # (60, N, 5)

        peak_I = snaps[:, :, 2].sum(dim=1).max().item()
        print(f'peak_I={peak_I:.0f}')

        if peak_I < 30:
            print('    WARNING: very low spread — skipping this pair')
            continue

        x0 = snaps[0]       # (N, 5) day 0
        y  = snaps[1:31]    # (30, N, 5) days 1-30

        all_pairs.append({
            'x0':         x0,
            'y':          y,
            'graph_seed': seed,
            'sim_idx':    sim_idx,
            'beta':       beta,
        })

print(f'\nTotal training pairs collected: {len(all_pairs)}')

In [ ]:
# Cell 7 — Plot a few curves to verify diversity
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

sample_indices = [0, 5, 10, 20, 30, 40]
labels = ['S','E','I','R','D']
colors = ['steelblue','orange','red','green','gray']

for ax, idx in zip(axes, sample_indices):
    if idx >= len(all_pairs):
        ax.axis('off'); continue
    pair   = all_pairs[idx]
    y      = pair['y']             # (30, N, 5)
    totals = y.sum(dim=1) / pair['y'].shape[1]  # (30, 5) mean fraction
    for i, (lbl, col) in enumerate(zip(labels, colors)):
        ax.plot(range(1, 31), totals[:, i].numpy(), label=lbl, color=col, linewidth=1.5)
    ax.set_title(f"seed={pair['graph_seed']} sim={pair['sim_idx']} β={pair['beta']}")
    ax.set_xlabel('Day'); ax.set_ylabel('Fraction')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Sample SEIRD curves — verifying diversity across graphs', fontsize=13)
plt.tight_layout()
plt.savefig(f'{BASE}/data/seird_diversity_check.png', dpi=120)
plt.show()
print('If the curves look different from each other, diversity is good.')

In [ ]:
# Cell 8 — Save training pairs
torch.save(all_pairs, f'{BASE}/data/training_pairs.pt')
print(f'Saved {len(all_pairs)} training pairs to {BASE}/data/training_pairs.pt')

# Also save metadata summary
import json
meta = [{
    'graph_seed': p['graph_seed'],
    'sim_idx':    p['sim_idx'],
    'beta':       p['beta'],
    'peak_I':     float(p['y'][:, :, 2].sum(dim=1).max()),
} for p in all_pairs]

with open(f'{BASE}/data/training_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Metadata saved.')